# DataFrame Acceleration: hipDF and hipDF-pandas on AMD GPUs

## Importing cudf and load the dataset

In [ ]:
import cudf

In [ ]:
dataset = cudf.read_parquet('transactions_synthetic_data.parquet')
display(type(dataset))
display(dataset.shape)
dataset.head()

## hipDF Series User Defined Functions (UDFs)

In [ ]:
# TransactionType to lower
dataset['TransactionType'] = dataset['TransactionType'].apply(lambda x:x.lower())
display(type(dataset))
dataset.head(3)

In [ ]:
# Transform CustomerGender to zero & one
def customer_gender_to_binary(x):
    return 1 if x == 'Male' else 0

dataset['CustomerGender'] = dataset['CustomerGender'].apply(customer_gender_to_binary)
display(type(dataset))
dataset.head(3)

In [ ]:
# Operate using data from multiple columns
def compute_transaction_amount_over_income(row):
    return row['TransactionAmount']/row['CustomerIncome']

dataset['transaction_amount_over_income'] = dataset.apply(compute_transaction_amount_over_income, axis = 1)
display(type(dataset))
dataset.head(3)

## User Defined Aggregations using GroupBy

In [ ]:
# Group by TransactionType and MerchantCategory
agg1 = dataset.groupby(['TransactionType','MerchantCategory']).agg({
    'TransactionAmount': ['sum', 'mean', 'max', 'min', 'std'],
    'TransactionFee': ['sum', 'mean', 'max', 'min', 'std']
}).reset_index()

display(type(agg1))
agg1.sort_values(by = ['TransactionType','MerchantCategory'])

# Use cudf.pandas acceleration layer

### Timing groupby operations on CPU

In [ ]:
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import pandas as pd

In [ ]:
dataset = pd.read_parquet('transactions_synthetic_data.parquet')
display(type(dataset))
dataset.head()

In [ ]:
%%time
# Group by TransactionType, CustomerRegion, AccountType and BranchCode
agg1 = dataset.groupby(['TransactionType', 'CustomerRegion','AccountType', 'BranchCode']).agg({
    'TransactionAmount': ['sum', 'mean', 'max', 'min', 'std'],
    'TransactionFee': ['sum', 'mean', 'max', 'min', 'std']
}).reset_index()

In [ ]:
type(pd.DataFrame.groupby)

In [ ]:
agg1.head()

### Timing groupby operations with cudf.pandas enabled

In [ ]:
# We need to set HSA_XNACK in cudf.pandas accelerator mode. 
# This setting enables page migrations between host and device.
%env HSA_XNACK=1

In [ ]:
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
%load_ext cudf.pandas

In [ ]:
import pandas as pd

dataset = pd.read_parquet('transactions_synthetic_data.parquet')

In [ ]:
%%time
# Group by TransactionType, CustomerRegion, AccountType and BranchCode
agg1 = dataset.groupby(['TransactionType', 'CustomerRegion','AccountType', 'BranchCode']).agg({
    'TransactionAmount': ['sum', 'mean', 'max', 'min', 'std'],
    'TransactionFee': ['sum', 'mean', 'max', 'min', 'std']
}).reset_index()

In [ ]:
%%time
%%cudf.pandas.profile

#Group by TransactionType and CustomerRegion, aggregate TransactionAmount and TransactionFee
agg1 = dataset.groupby(['TransactionType', 'CustomerRegion','AccountType', 'BranchCode']).agg({
    'TransactionAmount': ['sum', 'mean', 'max', 'min', 'std'],
    'TransactionFee': ['sum', 'mean', 'max', 'min', 'std']
}).reset_index()

In [ ]:
type(pd.DataFrame.groupby)

### Aggregation using pivot table

In [ ]:
%%time
# Pivot table with CustomerMaritalStatus and MerchantCategory, showing mean and sum of TransactionAmount
pivot1 = pd.pivot_table(dataset, values='TransactionAmount', index='CustomerMaritalStatus', columns='MerchantCategory', 
                        aggfunc=['mean', 'sum'])

In [ ]:
pivot1.head()

### Profiling pivot table execution

In [ ]:
%%time
%%cudf.pandas.profile
# Pivot table with CustomerMaritalStatus and MerchantCategory, showing mean and sum of TransactionAmount
pivot1 = pd.pivot_table(dataset, values='TransactionAmount', index='CustomerMaritalStatus', columns='MerchantCategory', 
                        aggfunc=['mean', 'sum'])

### Using groupby instead of pivot_table

In [ ]:
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
%load_ext cudf.pandas

In [ ]:
import pandas as pd

dataset = pd.read_parquet('transactions_synthetic_data.parquet')

In [ ]:
%%time
# Alternative to pivot_table using groupby
pivot1 = dataset.groupby(['CustomerMaritalStatus','MerchantCategory'])['TransactionAmount'].agg(['mean','sum']).unstack()

In [ ]:
pivot1.head()

In [ ]:
%%time
%%cudf.pandas.profile
# Alternative to pivot using groupby
pivot1 = dataset.groupby(['CustomerMaritalStatus','MerchantCategory'])['TransactionAmount'].agg(['mean','sum']).unstack()

## Appendix

In [ ]:
import plotly.graph_objects as go

categories = ["pandas CPU time", "hipDF GPU time"]
times = [2.02, 0.13]

fig = go.Figure(data = [
    go.Bar(name = 'Time', x = categories, y = times)
])

fig.update_layout(
    title = "Operation: groupby. Execution time: CPU vs GPU",
    xaxis_title = 'Category',
    yaxis_title = 'Time (seconds)',
    template = 'plotly_white',
    height = 500,
    width = 500
)